In [7]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [3]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [4]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [5]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [8]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [9]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS Account ID Extractor\n\nHere's a comprehensive solution with explanations and test cases:\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str | None:\n    \"\"\"\n    Extract the AWS account ID from an ARN string.\n    \n    AWS ARN format: arn:partition:service:region:account-id:resource-type/resource-id\n    \n    Args:\n        arn: An AWS ARN string\n        \n    Returns:\n        The AWS account ID (12-digit string) if present, None otherwise\n        \n    Raises:\n        ValueError: If the input is not a valid ARN format\n    \"\"\"\n    if not isinstance(arn, str):\n        raise ValueError(\"ARN must be a string\")\n    \n    if not arn.startswith(\"arn:\"):\n        raise ValueError(\"Invalid ARN format: must start with 'arn:'\")\n    \n    # Split the ARN by colons\n    parts = arn.split(\":\")\n    \n    # ARN format: arn:partition:service:region:account-id:resource-type/resource-id\n    # Minimum valid ARN has 6 parts (account-id cou